In [0]:
-- 步骤2: 保存到Silver层（Delta格式，外部表）
-- 设置当前会话的默认 catalog
-- USE CATALOG aws3;

-- 设置当前会话的默认 schema
-- USE SCHEMA silver;

-- 现在可以直接查询表
-- SELECT * FROM orders_info;  -- 相当于 aws3.silver.orders_info

--指定列的默认值要启用对应的表功能未。请先执行
ALTER TABLE tableName SET
TBLPROPERTIES('delta.feature.allowColumnDefaults' = 'supported')
--或者在建表属性加
TBLPROPERTIES('delta.feature.allowColumnDefaults' = 'supported')

-----bronze，不用指定LOCATION，已经在SCHEMA上指定了文件夹，用内部表，TBLPROPERTIES按AI的指定每层属性，默认DELTA表
CREATE OR REPLACE TABLE aws3.bronze.orders_info
(
    _bronze_load_ts TIMESTAMP,
    _bronze_load_id STRING,
    _source_system STRING,
    _source_table STRING,
    id                     BIGINT,
    consignee              STRING,
    consignee_tel          STRING,
    total_amount           DECIMAL(10,2),
    order_status           STRING,
    user_id                BIGINT,
    payment_way            STRING,
    delivery_address       STRING,
    order_comment          STRING,
    out_trade_no           STRING,
    trade_body             STRING,
    create_time            TIMESTAMP,
    operate_time           TIMESTAMP,
    expire_time            TIMESTAMP,
    process_status         STRING,
    tracking_no            STRING,
    parent_order_id        BIGINT,
    img_url                STRING,
    province_id            INT,
    activity_reduce_amount DECIMAL(16,2),
    coupon_reduce_amount   DECIMAL(16,2),
    original_total_amount  DECIMAL(16,2),
    feight_fee             DECIMAL(16,2),
    feight_fee_reduce      DECIMAL(16,2),
    refundable_time        TIMESTAMP,
    -- 分区列
    _bronze_load_date DATE GENERATED ALWAYS AS (CAST(_bronze_load_ts AS DATE))
)
USING DELTA
PARTITIONED BY (_bronze_load_date)
LOCATION 's3://aws-s3-cpuhy/databricks/bronze/orders_info'
COMMENT '原始订单数据 - 来自 MySQL'
TBLPROPERTIES (
    'delta.autoOptimize.optimizeWrite' = 'false',
    'delta.enableChangeDataFeed' = 'true',
    'delta.logRetentionDuration' = 'interval 7 days',
    'delta.deletedFileRetentionDuration' = 'interval 1 days',
    'delta.dataSkippingNumIndexedCols' = '3',
    'bronze.retention.days' = '365'   
);


CREATE OR REPLACE TABLE aws3.bronze.order_detail
(
    _bronze_load_ts TIMESTAMP,
    _bronze_load_id STRING,
    _source_system STRING,
    _source_table STRING,
    id                    bigint ,
    order_id              bigint        ,
    sku_id                bigint         ,
    sku_name              STRING   ,
    img_url               STRING   ,
    order_price           decimal(10, 2),
    sku_num               bigint         ,
    create_time           TIMESTAMP       ,
    split_total_amount    decimal(16, 2) ,
    split_activity_amount decimal(16, 2) ,
    split_coupon_amount   decimal(16, 2) ,
    operate_time          TIMESTAMP,
    -- 分区列
    _bronze_load_date DATE GENERATED ALWAYS AS (CAST(_bronze_load_ts AS DATE))
)
USING DELTA
PARTITIONED BY (_bronze_load_date)
LOCATION 's3://aws-s3-cpuhy/databricks/bronze/order_detail'
COMMENT '原始订单明细数据 - 来自 MySQL'
TBLPROPERTIES (
    'delta.autoOptimize.optimizeWrite' = 'false',
    'delta.enableChangeDataFeed' = 'true',
    'delta.logRetentionDuration' = 'interval 7 days',
    'delta.deletedFileRetentionDuration' = 'interval 1 days',
    'delta.dataSkippingNumIndexedCols' = '3',
    'bronze.retention.days' = '365'   
);


CREATE OR REPLACE TABLE  aws3.bronze.base_category1
(   
    _bronze_load_ts TIMESTAMP,
    _bronze_load_id STRING,
    _source_system STRING,
    _source_table STRING,
    id           bigint,
    name         STRING ,
    create_time  TIMESTAMP   ,
    operate_time TIMESTAMP,
    -- 分区列
    _bronze_load_date DATE GENERATED ALWAYS AS (CAST(_bronze_load_ts AS DATE))
)
USING DELTA
PARTITIONED BY (_bronze_load_date)
LOCATION 's3://aws-s3-cpuhy/databricks/bronze/base_category1'
TBLPROPERTIES (
    'delta.autoOptimize.optimizeWrite' = 'false',
    'delta.enableChangeDataFeed' = 'true',
    'delta.logRetentionDuration' = 'interval 7 days',
    'delta.deletedFileRetentionDuration' = 'interval 1 days',
    'delta.dataSkippingNumIndexedCols' = '3'
);

CREATE OR REPLACE TABLE  aws3.bronze.base_category2
(   
    _bronze_load_ts TIMESTAMP,
    _bronze_load_id STRING,
    _source_system STRING,
    _source_table STRING,
    id           bigint,
    name         STRING ,
    category1_id bigint,
    create_time  TIMESTAMP   ,
    operate_time TIMESTAMP,
    -- 分区列
    _bronze_load_date DATE GENERATED ALWAYS AS (CAST(_bronze_load_ts AS DATE))
)
USING DELTA
PARTITIONED BY (_bronze_load_date)
LOCATION 's3://aws-s3-cpuhy/databricks/bronze/base_category2'
TBLPROPERTIES (
    'delta.autoOptimize.optimizeWrite' = 'false',
    'delta.enableChangeDataFeed' = 'true',
    'delta.logRetentionDuration' = 'interval 7 days',
    'delta.deletedFileRetentionDuration' = 'interval 1 days',
    'delta.dataSkippingNumIndexedCols' = '3'
);

CREATE OR REPLACE TABLE  aws3.bronze.base_category3
(   
    _bronze_load_ts TIMESTAMP,
    _bronze_load_id STRING,
    _source_system STRING,
    _source_table STRING,
    id           bigint,
    name         STRING ,
    category2_id bigint,
    create_time  TIMESTAMP   ,
    operate_time TIMESTAMP,
    -- 分区列
    _bronze_load_date DATE GENERATED ALWAYS AS (CAST(_bronze_load_ts AS DATE))
)
USING DELTA
PARTITIONED BY (_bronze_load_date)
LOCATION 's3://aws-s3-cpuhy/databricks/bronze/base_category3'
TBLPROPERTIES (
    'delta.autoOptimize.optimizeWrite' = 'false',
    'delta.enableChangeDataFeed' = 'true',
    'delta.logRetentionDuration' = 'interval 7 days',
    'delta.deletedFileRetentionDuration' = 'interval 1 days',
    'delta.dataSkippingNumIndexedCols' = '3'
);

CREATE OR REPLACE TABLE  aws3.bronze.base_province
(   
    _bronze_load_ts TIMESTAMP,
    _bronze_load_id STRING,
    _source_system STRING,
    _source_table STRING,
    id           bigint      ,
    name         STRING ,
    region_id    STRING ,
    area_code    STRING ,
    iso_code     STRING ,
    iso_3166_2   STRING ,
    create_time  TIMESTAMP    ,
    operate_time TIMESTAMP,
    -- 分区列
    _bronze_load_date DATE GENERATED ALWAYS AS (CAST(_bronze_load_ts AS DATE))
)
USING DELTA
PARTITIONED BY (_bronze_load_date)
LOCATION 's3://aws-s3-cpuhy/databricks/bronze/base_province'
TBLPROPERTIES (
    'delta.autoOptimize.optimizeWrite' = 'false',
    'delta.enableChangeDataFeed' = 'true',
    'delta.logRetentionDuration' = 'interval 7 days',
    'delta.deletedFileRetentionDuration' = 'interval 1 days',
    'delta.dataSkippingNumIndexedCols' = '3'
);

CREATE OR REPLACE TABLE  aws3.bronze.base_region
(   
    _bronze_load_ts TIMESTAMP,
    _bronze_load_id STRING,
    _source_system STRING,
    _source_table STRING,
    id           bigint ,
    region_name  STRING ,
    create_time  TIMESTAMP    ,
    operate_time TIMESTAMP,
    -- 分区列
    _bronze_load_date DATE GENERATED ALWAYS AS (CAST(_bronze_load_ts AS DATE))
)
USING DELTA
PARTITIONED BY (_bronze_load_date)
LOCATION 's3://aws-s3-cpuhy/databricks/bronze/base_region'
TBLPROPERTIES (
    'delta.autoOptimize.optimizeWrite' = 'false',
    'delta.enableChangeDataFeed' = 'true',
    'delta.logRetentionDuration' = 'interval 7 days',
    'delta.deletedFileRetentionDuration' = 'interval 1 days',
    'delta.dataSkippingNumIndexedCols' = '3'
);


CREATE OR REPLACE TABLE  aws3.bronze.sku_info
(   
    _bronze_load_ts TIMESTAMP,
    _bronze_load_id STRING,
    _source_system STRING,
    _source_table STRING,
    id              bigint ,
    spu_id          bigint            ,
    price           decimal           ,
    sku_name        STRING     ,
    sku_desc        STRING     ,
    weight          decimal(10, 2)    ,
    tm_id           bigint           ,
    category3_id    bigint           ,
    sku_default_img STRING      ,
    is_sale         int ,
    create_time     TIMESTAMP          ,
    operate_time    TIMESTAMP,
    -- 分区列
    _bronze_load_date DATE GENERATED ALWAYS AS (CAST(_bronze_load_ts AS DATE))
)
USING DELTA
PARTITIONED BY (_bronze_load_date)
LOCATION 's3://aws-s3-cpuhy/databricks/bronze/sku_info'
TBLPROPERTIES (
    'delta.autoOptimize.optimizeWrite' = 'false',
    'delta.enableChangeDataFeed' = 'true',
    'delta.logRetentionDuration' = 'interval 7 days',
    'delta.deletedFileRetentionDuration' = 'interval 1 days',
    'delta.dataSkippingNumIndexedCols' = '3'
);



CREATE OR REPLACE TABLE  aws3.bronze.spu_info
(   
    _bronze_load_ts TIMESTAMP,
    _bronze_load_id STRING,
    _source_system STRING,
    _source_table STRING,
    id           bigint ,
    spu_name     STRING ,
    description  STRING,
    category3_id bigint       ,
    tm_id        bigint       ,
    create_time  TIMESTAMP     ,
    operate_time TIMESTAMP,
    -- 分区列
    _bronze_load_date DATE GENERATED ALWAYS AS (CAST(_bronze_load_ts AS DATE))
)
USING DELTA
PARTITIONED BY (_bronze_load_date)
LOCATION 's3://aws-s3-cpuhy/databricks/bronze/spu_info'
TBLPROPERTIES (
    'delta.autoOptimize.optimizeWrite' = 'false',
    'delta.enableChangeDataFeed' = 'true',
    'delta.logRetentionDuration' = 'interval 7 days',
    'delta.deletedFileRetentionDuration' = 'interval 1 days',
    'delta.dataSkippingNumIndexedCols' = '3'
);


CREATE OR REPLACE TABLE  aws3.bronze.user_info
(   
    _bronze_load_ts TIMESTAMP,
    _bronze_load_id STRING,
    _source_system STRING,
    _source_table STRING,
    id           bigint ,
    login_name   STRING ,
    nick_name    STRING ,
    passwd       STRING ,
    name         STRING ,
    phone_num    STRING ,
    email        STRING ,
    head_img     STRING ,
    user_level   STRING ,
    birthday     date         ,
    gender       STRING  ,
    create_time  TIMESTAMP     ,
    operate_time TIMESTAMP     ,
    status       STRING,
    -- 分区列
    _bronze_load_date DATE GENERATED ALWAYS AS (CAST(_bronze_load_ts AS DATE))
)
USING DELTA
PARTITIONED BY (_bronze_load_date)
LOCATION 's3://aws-s3-cpuhy/databricks/bronze/user_info'
TBLPROPERTIES (
    'delta.autoOptimize.optimizeWrite' = 'false',
    'delta.enableChangeDataFeed' = 'true',
    'delta.logRetentionDuration' = 'interval 7 days',
    'delta.deletedFileRetentionDuration' = 'interval 1 days',
    'delta.dataSkippingNumIndexedCols' = '3'
);

CREATE OR REPLACE TABLE  aws3.bronze.base_trademark
(   
    _bronze_load_ts TIMESTAMP,
    _bronze_load_id STRING,
    _source_system STRING,
    _source_table STRING,
    id           bigint ,
    tm_name  STRING ,
    logo_url    STRING,
    create_time  TIMESTAMP    ,
    operate_time TIMESTAMP,
    -- 分区列
    _bronze_load_date DATE GENERATED ALWAYS AS (CAST(_bronze_load_ts AS DATE))
)
USING DELTA
PARTITIONED BY (_bronze_load_date)
LOCATION 's3://aws-s3-cpuhy/databricks/bronze/base_trademark'
TBLPROPERTIES (
    'delta.autoOptimize.optimizeWrite' = 'false',
    'delta.enableChangeDataFeed' = 'true',
    'delta.logRetentionDuration' = 'interval 7 days',
    'delta.deletedFileRetentionDuration' = 'interval 1 days',
    'delta.dataSkippingNumIndexedCols' = '3'
);


drop table aws3.silver.dim_user

----silver
-- SILVER: 客户维度表 (SCD Type 2)
CREATE OR REPLACE TABLE aws3.silver.dim_user (
    customer_key BIGINT GENERATED ALWAYS AS IDENTITY,
    id           bigint ,
    login_name   STRING ,
    nick_name    STRING ,
    passwd       STRING ,
    name         STRING ,
    phone_num    STRING ,
    email        STRING ,
    head_img     STRING ,
    user_level   STRING ,
    birthday     date         ,
    gender       STRING  ,
    create_time  TIMESTAMP     ,
    operate_time TIMESTAMP     ,
    status       STRING,
    -- SCD Type 2 字段
    valid_from TIMESTAMP, --NOT NULL,
    valid_to TIMESTAMP,
    is_current BOOLEAN DEFAULT true,
    -- 元数据
    source_system STRING,
    --created_at TIMESTAMP,用回原字段create_time，operate_time
    --updated_at TIMESTAMP,
    load_batch_id STRING,
    -- 约束
    CONSTRAINT pk_dim_user PRIMARY KEY(customer_key, valid_from)
)
USING DELTA
LOCATION 's3://aws-s3-cpuhy/databricks/silver/dim_user'
COMMENT '用户维度表 - SCD Type 2'
TBLPROPERTIES (
    'delta.autoOptimize.optimizeWrite' = 'true',
    'delta.enableChangeDataFeed' = 'true',
    'delta.logRetentionDuration' = 'interval 90 days',
    'delta.deletedFileRetentionDuration' = 'interval 7 days',
    'delta.dataSkippingNumIndexedCols' = '8',
    --'delta.constraints.violations' = 'drop',
    'silver.data.quality.threshold' = '0.95',
    'delta.feature.allowColumnDefaults' = 'supported'  --列默认值
);


-- SILVER: 地区维度表
CREATE OR REPLACE TABLE aws3.silver.dim_province 
USING DELTA
LOCATION 's3://aws-s3-cpuhy/databricks/silver/dim_province'
TBLPROPERTIES (
    'delta.autoOptimize.optimizeWrite' = 'true',  -- 开启，因为每天全量写入
    'delta.autoOptimize.autoCompact' = 'true',
    'delta.dataSkippingNumIndexedCols' = '10',  -- 提高数据跳过效率
    -- 全量表可以保留更短的日志
    'delta.logRetentionDuration' = 'interval 7 days'
)
AS
select
    province.id,
    province.name,
    province.area_code,
    province.iso_code,
    province.iso_3166_2,
    region_id,
    region_name,
    CURRENT_TIMESTAMP() as load_time
from
(
    select
        id,
        name,
        region_id,
        area_code,
        iso_code,
        iso_3166_2
    from aws3.bronze.base_province
)province
left join
(
    select
        id,
        region_name
    from aws3.bronze.base_region
)region
on province.region_id=region.id;


-- SILVER: 产品维度表
CREATE OR REPLACE TABLE aws3.silver.dim_sku 
USING DELTA
LOCATION 's3://aws-s3-cpuhy/databricks/silver/dim_sku'
TBLPROPERTIES (
    'delta.autoOptimize.optimizeWrite' = 'true',  -- 开启，因为每天全量写入
    'delta.autoOptimize.autoCompact' = 'true',
    'delta.dataSkippingNumIndexedCols' = '10',  -- 提高数据跳过效率
    -- 全量表可以保留更短的日志
    'delta.logRetentionDuration' = 'interval 7 days'
)
AS
with
sku as
(
    select
        id,
        price,
        sku_name,
        sku_desc,
        weight,
        is_sale,
        spu_id,
        category3_id,
        tm_id,
        create_time
    from aws3.bronze.sku_info
),
spu as
(
    select
        id,
        spu_name
    from aws3.bronze.spu_info
),
c3 as
(
    select
        id,
        name,
        category2_id
    from aws3.bronze.base_category3
),
c2 as
(
    select
        id,
        name,
        category1_id
    from aws3.bronze.base_category2
),
c1 as
(
    select
        id,
        name
    from aws3.bronze.base_category1
),
tm as
(
    select
        id,
        tm_name
    from aws3.bronze.base_trademark
)
select
    sku.id,
    sku.price,
    sku.sku_name,
    sku.sku_desc,
    sku.weight,
    sku.is_sale,
    sku.spu_id,
    spu.spu_name,
    sku.category3_id,
    c3.name category3_name,
    c3.category2_id,
    c2.name category2_name,
    c2.category1_id,
    c1.name category1_name,
    sku.tm_id,
    tm.tm_name,
    CURRENT_TIMESTAMP() as load_time
from sku
left join spu on sku.spu_id=spu.id
left join c3 on sku.category3_id=c3.id
left join c2 on c3.category2_id=c2.id
left join c1 on c2.category1_id=c1.id
left join tm on sku.tm_id=tm.id
;


-- SILVER: 时间维度表
CREATE OR REPLACE TABLE aws3.silver.dim_date 
USING DELTA
LOCATION 's3://aws-s3-cpuhy/databricks/silver/dim_date'
TBLPROPERTIES (
    'delta.autoOptimize.optimizeWrite' = 'true',  -- 开启，因为每天全量写入
    'delta.autoOptimize.autoCompact' = 'true',
    'delta.dataSkippingNumIndexedCols' = '10',  -- 提高数据跳过效率
    -- 全量表可以保留更短的日志
    'delta.logRetentionDuration' = 'interval 7 days'
)
AS
WITH date_series AS (
    SELECT explode(sequence(
        to_date('2020-01-01'), 
        to_date('2030-12-31'), 
        interval 1 day
    )) as date
)
SELECT 
    -- 代理键
    CAST(date_format(date, 'yyyyMMdd') AS INT) as date_key,
    
    -- 日期属性
    date as full_date,
    YEAR(date) as year,
    QUARTER(date) as quarter,
    MONTH(date) as month,
    DAY(date) as day,
    
    -- 周信息
    WEEKOFYEAR(date) as week_of_year,
    DAYOFWEEK(date) as day_of_week,
    
    -- 业务标记
    CASE WHEN DAYOFWEEK(date) IN (1,7) THEN 1 ELSE 0 END as is_weekend,
    CASE WHEN MONTH(date) = 12 AND DAY(date) = 25 THEN 1 ELSE 0 END as is_christmas,
    CURRENT_TIMESTAMP() as load_time
    
FROM date_series;


-- SILVER: 订单事实表
CREATE OR REPLACE TABLE aws3.silver.fact_orders
USING DELTA
PARTITIONED BY (create_month)  --(DATE_TRUNC('MONTH', create_time))不支持表达式分区
LOCATION 's3://aws-s3-cpuhy/databricks/silver/fact_orders'
COMMENT '订单事实表'
TBLPROPERTIES (
    'delta.autoOptimize.optimizeWrite' = 'true',
    'delta.enableChangeDataFeed' = 'true',
    'delta.logRetentionDuration' = 'interval 30 days',
    'delta.deletedFileRetentionDuration' = 'interval 7 days',
    'delta.dataSkippingNumIndexedCols' = '12'
    --'delta.constraints.violations' = 'drop'
)
--CLUSTER BY (customer_key, order_status)--数据库在磁盘上存储数据时，按照 customer_key 和 order_status 这两个字段的顺序，将相同或相近值的数据行物理地排列在一起，查快
AS
select
    od.id,
    od.order_id,
    user_id,
    sku_id,
    province_id,
    CAST(DATE_FORMAT(od.create_time, 'yyyyMMdd') AS INT) as date_id,
    --to_number(to_char(od.create_time, 'yyyyMMdd'),99999999) as date_id,
    (DATE_TRUNC('MONTH', od.create_time)) as create_month,
    sku_num,
    order_price,
    sku_num * order_price split_original_amount,
    nvl(split_activity_amount,0.0) split_activity_amount,
    nvl(split_coupon_amount,0.0) split_coupon_amount,
    split_total_amount,
    case when COALESCE(od.operate_time,od.create_time) < COALESCE(oi.operate_time,oi.create_time) then COALESCE(oi.operate_time,oi.create_time) else COALESCE(od.operate_time,od.create_time) end as ods_max_update_time,
    CURRENT_TIMESTAMP() as load_time,
    CURRENT_TIMESTAMP() as update_time
from
    aws3.bronze.order_detail od
    left join aws3.bronze.orders_info oi on od.order_id = oi.id
;


--上次建表的参数，参照
TBLPROPERTIES (
    'delta.autoOptimize.optimizeWrite' = 'true',  -- 必须开启，增量写入
    'delta.autoOptimize.autoCompact' = 'true',
    'delta.dataSkippingNumIndexedCols' = '10',
    -- 事实表需要更长的日志保留（可能需要时间旅行）
    'delta.logRetentionDuration' = 'interval 90 days',
    -- 建议设置文件大小目标
    'delta.targetFileSize' = '134217728'  -- 128MB
)

select * from aws3.silver.fact_orders
-- SILVER: 


select * from aws3.gold.daily_sales_summary


---GOLD
-- GOLD: 每日销售汇总表
CREATE OR REPLACE TABLE aws3.gold.daily_sales_summary 
USING DELTA
PARTITIONED BY (full_date)
LOCATION 's3://aws-s3-cpuhy/databricks/gold/daily_sales_summary'
COMMENT '日报表：销售汇总'
TBLPROPERTIES (
    'delta.autoOptimize.optimizeWrite' = 'true',
    'delta.enableChangeDataFeed' = 'false',
    'delta.logRetentionDuration' = 'interval 180 days',
    'delta.deletedFileRetentionDuration' = 'interval 30 days',
    'delta.dataSkippingNumIndexedCols' = '6',
    'delta.targetFileSize' = '128MB',
    'gold.refresh.frequency' = 'daily'
)
--CLUSTER BY (province_name, category3_name)
AS
select
   dd.full_date,
   dd.day,
   dd.year,
   dd.quarter,
   dd.month,
   dd.is_weekend,
   dd.day_of_week,
   dd.week_of_year,
   dp.name province_name,
   dp.region_name,
   ds.sku_name,
   ds.spu_name,
   ds.category3_name,
   ds.category2_name,
   ds.category1_name,
   ds.tm_name,
   nvl(dus.name,'未知') user_name,--应该在维表时处理，以下都是
   nvl(dus.birthday,CAST('9999-12-31' AS DATE)) birthday,
   nvl(dus.gender,'未知') gender,
   nvl(dus.phone_num,'未知') phone_num,
   nvl(dus.email,'未知') email,
   nvl(dus.user_level,'未知') user_level,
   sum(f.sku_num) sku_num,
   sum(f.split_activity_amount) split_activity_amount,
   sum(f.split_coupon_amount) split_coupon_amount,
   sum(f.split_original_amount) split_original_amount,
   sum(f.split_total_amount) split_total_amount
from
    aws3.silver.fact_orders f
    left join aws3.silver.dim_date dd on f.date_id = dd.date_key
    left join aws3.silver.dim_province dp on f.province_id = dp.id
    left join aws3.silver.dim_sku ds on f.sku_id = ds.id
    left join aws3.silver.dim_user dus on f.user_id = dus.id 
        and dus.is_current = true  --当前状态的用户状态,两个只用一个
        --and f.create_time between dus.valid_from and dus.valid_to  --订单时的用户状态且是上线后的，因为没有之前状态,两个只用一个
group by 
   dd.full_date,
   dd.day,
   dd.year,
   dd.quarter,
   dd.month,
   dd.is_weekend,
   dd.day_of_week,
   dd.week_of_year,
   dp.name,
   dp.region_name,
   ds.sku_name,
   ds.spu_name,
   ds.category3_name,
   ds.category2_name,
   ds.category1_name,
   ds.tm_name,
   nvl(dus.name,'未知'),--应该在维表时处理，以下都是
   nvl(dus.birthday,CAST('9999-12-31' AS DATE)),
   nvl(dus.gender,'未知'),
   nvl(dus.phone_num,'未知'),
   nvl(dus.email,'未知'),
   nvl(dus.user_level,'未知')
;



In [0]:
-- 选项1: 删除所有表（注意：这会删除表和数据）
DROP TABLE IF EXISTS aws3.silver.orders_info;


In [0]:
-- 选项2: 使用 ALTER TABLE 添加列（保留现有数据）

-- aws3.silver.base_category1 添加列
ALTER TABLE aws3.silver.base_category1 ADD COLUMNS (
    source_file STRING,
    load_timestamp TIMESTAMP,
    update_timestamp TIMESTAMP
);



In [0]:
-- aws3.silver.user_info 添加列
ALTER TABLE aws3.silver.user_info ADD COLUMNS (
    is_latest BOOLEAN
);

update aws3.silver.user_info set is_latest = true;

select * from aws3.silver.user_info

In [0]:
--保存到glod层（Delta格式,内部表）

CREATE TABLE adhyivy.default.gold_dim_date
USING DELTA
-- LOCATION 'abfss://bronze@sahyivy.dfs.core.windows.net/gold/dim_date'
TBLPROPERTIES (
    'delta.autoOptimize.optimizeWrite' = 'true',  -- 开启，因为每天全量写入
    'delta.autoOptimize.autoCompact' = 'true',
    'delta.dataSkippingNumIndexedCols' = '10',  -- 提高数据跳过效率
    -- 全量表可以保留更短的日志
    'delta.logRetentionDuration' = 'interval 7 days'
)
AS
WITH date_series AS (
    SELECT explode(sequence(
        to_date('2020-01-01'), 
        to_date('2030-12-31'), 
        interval 1 day
    )) as date
)
SELECT 
    -- 代理键
    CAST(date_format(date, 'yyyyMMdd') AS INT) as date_key,
    
    -- 日期属性
    date as full_date,
    YEAR(date) as year,
    QUARTER(date) as quarter,
    MONTH(date) as month,
    DAY(date) as day,
    
    -- 周信息
    WEEKOFYEAR(date) as week_of_year,
    DAYOFWEEK(date) as day_of_week,
    
    -- 业务标记
    CASE WHEN DAYOFWEEK(date) IN (1,7) THEN 1 ELSE 0 END as is_weekend,
    CASE WHEN MONTH(date) = 12 AND DAY(date) = 25 THEN 1 ELSE 0 END as is_christmas,
    CURRENT_TIMESTAMP() as load_time
    
FROM date_series;


CREATE TABLE adhyivy.default.gold_dim_province 
USING DELTA
-- LOCATION 'abfss://bronze@sahyivy.dfs.core.windows.net/gold/dim_province'
TBLPROPERTIES (
    'delta.autoOptimize.optimizeWrite' = 'true',  -- 开启，因为每天全量写入
    'delta.autoOptimize.autoCompact' = 'true',
    'delta.dataSkippingNumIndexedCols' = '10',  -- 提高数据跳过效率
    -- 全量表可以保留更短的日志
    'delta.logRetentionDuration' = 'interval 7 days'
)
AS
select
    province.id,
    province.name,
    province.area_code,
    province.iso_code,
    province.iso_3166_2,
    region_id,
    region_name,
    CURRENT_TIMESTAMP() as load_time
from
(
    select
        id,
        name,
        region_id,
        area_code,
        iso_code,
        iso_3166_2
    from aws3.silver.base_province
)province
left join
(
    select
        id,
        region_name
    from aws3.silver.base_region
)region
on province.region_id=region.id;


CREATE TABLE adhyivy.default.gold_dim_sku 
USING DELTA
--LOCATION 'abfss://bronze@sahyivy.dfs.core.windows.net/gold/dim_sku'
TBLPROPERTIES (
    'delta.autoOptimize.optimizeWrite' = 'true',  -- 开启，因为每天全量写入
    'delta.autoOptimize.autoCompact' = 'true',
    'delta.dataSkippingNumIndexedCols' = '10',  -- 提高数据跳过效率
    -- 全量表可以保留更短的日志
    'delta.logRetentionDuration' = 'interval 7 days'
)
AS
with
sku as
(
    select
        id,
        price,
        sku_name,
        sku_desc,
        weight,
        is_sale,
        spu_id,
        category3_id,
        tm_id,
        create_time
    from aws3.silver.sku_info
),
spu as
(
    select
        id,
        spu_name
    from aws3.silver.spu_info
),
c3 as
(
    select
        id,
        name,
        category2_id
    from aws3.silver.base_category3
),
c2 as
(
    select
        id,
        name,
        category1_id
    from aws3.silver.base_category2
),
c1 as
(
    select
        id,
        name
    from aws3.silver.base_category1
),
tm as
(
    select
        id,
        tm_name
    from aws3.silver.base_trademark
)
select
    sku.id,
    sku.price,
    sku.sku_name,
    sku.sku_desc,
    sku.weight,
    sku.is_sale,
    sku.spu_id,
    spu.spu_name,
    sku.category3_id,
    c3.name category3_name,
    c3.category2_id,
    c2.name category2_name,
    c2.category1_id,
    c1.name category1_name,
    sku.tm_id,
    tm.tm_name,
    CURRENT_TIMESTAMP() as load_time
from sku
left join spu on sku.spu_id=spu.id
left join c3 on sku.category3_id=c3.id
left join c2 on c3.category2_id=c2.id
left join c1 on c2.category1_id=c1.id
left join tm on sku.tm_id=tm.id
;





--建表SQL
CREATE TABLE adhyivy.default.gold_fact_sales
USING DELTA
 -- LOCATION 'abfss://bronze@sahyivy.dfs.core.windows.net/gold/fact_sales'
TBLPROPERTIES (
    'delta.autoOptimize.optimizeWrite' = 'true',  -- 必须开启，增量写入
    'delta.autoOptimize.autoCompact' = 'true',
    'delta.dataSkippingNumIndexedCols' = '10',
    -- 事实表需要更长的日志保留（可能需要时间旅行）
    'delta.logRetentionDuration' = 'interval 90 days',
    -- 建议设置文件大小目标
    'delta.targetFileSize' = '134217728'  -- 128MB
)
AS
select
    od.id,
    od.order_id,
    user_id,
    sku_id,
    province_id,
    to_number(to_char(od.create_time, 'yyyyMMdd'),99999999) as date_id,
    od.create_time,
    sku_num,
    order_price,
    sku_num * order_price split_original_amount,
    nvl(split_activity_amount,0.0) split_activity_amount,
    nvl(split_coupon_amount,0.0) split_coupon_amount,
    split_total_amount,
    case when od.update_timestamp < oi.update_timestamp then oi.update_timestamp else od.update_timestamp end as ods_max_update_time,
    CURRENT_TIMESTAMP() as load_time,
    CURRENT_TIMESTAMP() as update_time
from
    adhyivy.default.aws3.silver.order_detail od
    left join adhyivy.default.aws3.silver.orders_info oi on od.order_id = oi.id
;